In [1]:
# ==============================================================================
# 🔒 TINYLLAMA CoT — CLEAN KERNEL
# ==============================================================================

import sys
import torch
import transformers

print("=" * 100)
print("🔒 TINYLLAMA CoT — CLEAN KERNEL")
print("=" * 100)

print(f"Python       : {sys.version}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM         : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print("=" * 100)
print("✅ CLEAN KERNEL READY")
print("=" * 100)

🔒 TINYLLAMA CoT — CLEAN KERNEL
Python       : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch      : 2.6.0+cu124
Transformers : 5.12.1
CUDA         : 12.4
CUDA available: True
GPU          : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM         : 6.00 GB
✅ CLEAN KERNEL READY


In [2]:
# ==============================================================================
# 🔒 STEP 2 — TINYLLAMA TOKENIZER CPU TEST
# ==============================================================================

from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("=" * 100)
print("🔒 TINYLLAMA TOKENIZER — CPU TEST")
print("=" * 100)

print(f"Model: {MODEL_NAME}")
print("Loading tokenizer from local cache only...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    use_fast=False
)

print("\n✅ TOKENIZER LOADED")
print(f"Tokenizer class: {tokenizer.__class__.__name__}")
print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"EOS token      : {repr(tokenizer.eos_token)}")
print(f"PAD token      : {repr(tokenizer.pad_token)}")

# Simple stability test
test_text = "This is a test sentence."

tokens = tokenizer(
    test_text,
    return_tensors="pt"
)

decoded = tokenizer.decode(
    tokens["input_ids"][0],
    skip_special_tokens=False
)

print(f"\nToken shape: {tokens['input_ids'].shape}")
print(f"Decoded: {decoded}")

print("\n" + "=" * 100)
print("✅ TOKENIZER TEST PASSED")
print("=" * 100)

🔒 TINYLLAMA TOKENIZER — CPU TEST
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Loading tokenizer from local cache only...

✅ TOKENIZER LOADED
Tokenizer class: LlamaTokenizer
Vocab size     : 32000
EOS token      : '</s>'
PAD token      : '</s>'

Token shape: torch.Size([1, 7])
Decoded: <s> This is a test sentence.

✅ TOKENIZER TEST PASSED


In [3]:
# ==============================================================================
# 🔥 STEP 3 — TINYLLAMA + LoRA CHECKPOINT-4300
# ==============================================================================

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

LORA_CHECKPOINT = (
    r"C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300"
)

print("=" * 100)
print("🔥 LOADING TINYLLAMA CHECKPOINT-4300")
print("=" * 100)

print(f"\nBase model:")
print(MODEL_NAME)

print(f"\nLoRA checkpoint:")
print(LORA_CHECKPOINT)

if not os.path.exists(LORA_CHECKPOINT):
    raise FileNotFoundError(
        f"❌ LoRA checkpoint not found:\n{LORA_CHECKPOINT}"
    )

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

gc.collect()

print(
    f"\nVRAM before loading: "
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
)

# ==============================================================================
# LOAD BASE MODEL ON CPU FIRST
# ==============================================================================

print("\n🔵 Loading base TinyLlama on CPU...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("✅ Base TinyLlama loaded")

# ==============================================================================
# MOVE BASE MODEL TO GPU
# ==============================================================================

print("\n🔵 Moving base model to GPU...")

base_model = base_model.to("cuda")

print("✅ Base model moved to GPU")

print(
    f"VRAM after base: "
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
)

# ==============================================================================
# LOAD LoRA
# ==============================================================================

print("\n🔵 Loading LoRA checkpoint-4300...")

model = PeftModel.from_pretrained(
    base_model,
    LORA_CHECKPOINT,
    is_trainable=False
)

print("✅ LoRA checkpoint loaded successfully")

# ==============================================================================
# EVALUATION MODE
# ==============================================================================

model.eval()

for param in model.parameters():
    param.requires_grad_(False)

print("\nTraining: OFF")
print("Gradients: OFF")
print("Evaluation mode:", not model.training)
print("Device:", next(model.parameters()).device)

print(
    f"VRAM after LoRA: "
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
)

print("\n" + "=" * 100)
print("🎉 TINYLLAMA CoT MODEL READY")
print("=" * 100)

print("⚠️ NO GENERATION PERFORMED")
print("=" * 100)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


🔥 LOADING TINYLLAMA CHECKPOINT-4300

Base model:
TinyLlama/TinyLlama-1.1B-Chat-v1.0

LoRA checkpoint:
C:\Users\mohsi\tinyllama-plainlang-rtx3060\checkpoint-4300

VRAM before loading: 0.000 GB

🔵 Loading base TinyLlama on CPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base TinyLlama loaded

🔵 Moving base model to GPU...
✅ Base model moved to GPU
VRAM after base: 2.051 GB

🔵 Loading LoRA checkpoint-4300...
✅ LoRA checkpoint loaded successfully

Training: OFF
Gradients: OFF
Evaluation mode: True
Device: cuda:0
VRAM after LoRA: 2.098 GB

🎉 TINYLLAMA CoT MODEL READY
⚠️ NO GENERATION PERFORMED


In [4]:
# ==============================================================================
# 🔒 STEP 4 — RESTORE ASSET VALIDATION + TEST
# ==============================================================================

import os
import pyarrow as pa
import pyarrow.ipc as ipc

VALIDATION_FILE = (
    r"C:\Users\mohsi\.cache\huggingface\datasets"
    r"\facebook___asset\simplification\0.0.0"
    r"\c7f2fa4bae55ae656091805d4416c1374582bb4e"
    r"\asset-validation.arrow"
)

TEST_FILE = (
    r"C:\Users\mohsi\.cache\huggingface\datasets"
    r"\facebook___asset\simplification\0.0.0"
    r"\c7f2fa4bae55ae656091805d4416c1374582bb4e"
    r"\asset-test.arrow"
)

print("=" * 100)
print("🔒 RESTORING ASSET — PURE PYARROW")
print("=" * 100)

# ==============================================================================
# HELPER
# ==============================================================================

def load_asset_arrow(path):

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ File not found:\n{path}"
        )

    print(f"\nReading:")
    print(path)

    with pa.memory_map(path, "r") as source:

        reader = ipc.open_stream(source)

        table = reader.read_all()

    return table


# ==============================================================================
# VALIDATION
# ==============================================================================

validation_table = load_asset_arrow(
    VALIDATION_FILE
)

print(
    f"Validation rows: {validation_table.num_rows}"
)

# ==============================================================================
# TEST
# ==============================================================================

test_table = load_asset_arrow(
    TEST_FILE
)

print(
    f"Test rows: {test_table.num_rows}"
)

# ==============================================================================
# CONVERT TO PYTHON
# ==============================================================================

validation_originals = (
    validation_table["original"].to_pylist()
)

validation_references = (
    validation_table["simplifications"].to_pylist()
)

test_originals = (
    test_table["original"].to_pylist()
)

test_references = (
    test_table["simplifications"].to_pylist()
)

# ==============================================================================
# COMBINE
# ==============================================================================

originals = (
    validation_originals +
    test_originals
)

reference_groups = (
    validation_references +
    test_references
)

# ==============================================================================
# NORMALIZE REFERENCES
# ==============================================================================

reference_groups = [
    list(refs)
    for refs in reference_groups
]

# ==============================================================================
# VERIFICATION
# ==============================================================================

print("\n" + "=" * 100)
print("🔍 ASSET VERIFICATION")
print("=" * 100)

print(
    f"Total samples          : {len(originals)}"
)

print(
    f"Reference groups       : {len(reference_groups)}"
)

total_references = sum(
    len(refs)
    for refs in reference_groups
)

print(
    f"Total human references : {total_references}"
)

print(
    f"References per sample  : "
    f"{len(reference_groups[0])}"
)

# ==============================================================================
# FIRST SAMPLE
# ==============================================================================

print("\n" + "=" * 100)
print("🧪 FIRST SAMPLE")
print("=" * 100)

print("\nOriginal:")
print(originals[0])

print("\nReferences:")

for i, ref in enumerate(
    reference_groups[0],
    start=1
):
    print(f"{i}. {ref}")

# ==============================================================================
# FINAL CHECKS
# ==============================================================================

assert len(originals) == 2359
assert len(reference_groups) == 2359

assert all(
    len(refs) == 10
    for refs in reference_groups
)

print("\n" + "=" * 100)
print("✅ ASSET RESTORED SUCCESSFULLY")
print("=" * 100)

print("\n" + "=" * 100)
print("✅ VARIABLES READY")
print("=" * 100)

print(
    f"originals         : {len(originals)}"
)

print(
    f"reference_groups  : {len(reference_groups)}"
)

print("=" * 100)

🔒 RESTORING ASSET — PURE PYARROW

Reading:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-validation.arrow
Validation rows: 2000

Reading:
C:\Users\mohsi\.cache\huggingface\datasets\facebook___asset\simplification\0.0.0\c7f2fa4bae55ae656091805d4416c1374582bb4e\asset-test.arrow
Test rows: 359

🔍 ASSET VERIFICATION
Total samples          : 2359
Reference groups       : 2359
Total human references : 23590
References per sample  : 10

🧪 FIRST SAMPLE

Original:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

References:
1. countries next to it are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.
2. Nearby counties are Marin, Mendocino, Lake, Napa, and Solano and Contra Costa.
3. Adjacent counties are Marin, Mendocino, Lake, Napa, Solano and Contra Costa.
4. Neighboring counties are Marin, Mendocin

In [5]:
# ==============================================================================
# 🔧 STEP 5 — TINYLLAMA CoT PROMPT + CLEAN PREDICTION
# ==============================================================================

import re

# ==============================================================================
# CoT PROMPT
# ==============================================================================

def cot_prompt(sentence):

    return f"""Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add new information.
Do not remove important information.

First, think carefully about the meaning of the sentence and identify
difficult words, phrases, or structures that should be simplified.

Then produce the final simplified sentence.

Do not show your reasoning.
Return only the final simplified sentence.

Original: {sentence}

Simplified:"""


# ==============================================================================
# CLEAN PREDICTION
# ==============================================================================

def clean_prediction(text):

    if text is None:
        return ""

    text = str(text).strip()

    # --------------------------------------------------------------------------
    # Remove special tokens
    # --------------------------------------------------------------------------

    special_tokens = [
        "<|endoftext|>",
        "<|user|>",
        "<|assistant|>",
        "<s>",
        "</s>",
        "<|system|>",
    ]

    for token in special_tokens:
        text = text.replace(token, " ")

    # --------------------------------------------------------------------------
    # Remove common prompt artifacts
    # --------------------------------------------------------------------------

    # Keep only content after the final "Simplified:" if the model
    # repeated the prompt.
    if "Simplified:" in text:
        parts = text.split("Simplified:")

        if len(parts) > 1:
            text = parts[-1].strip()

    # Remove accidental "Original:" sections
    if "Original:" in text:
        text = text.split("Original:")[0].strip()

    # --------------------------------------------------------------------------
    # Remove user/assistant remnants
    # --------------------------------------------------------------------------

    text = re.split(
        r"<\|user\|>|<\|assistant\|>|<\|system\|>",
        text,
        flags=re.IGNORECASE
    )[0].strip()

    # --------------------------------------------------------------------------
    # Remove leading labels
    # --------------------------------------------------------------------------

    text = re.sub(
        r"^(Answer|Response|Simplified|Output)\s*:\s*",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    # --------------------------------------------------------------------------
    # Remove repeated identical sentences
    #
    # Example:
    # "The sentence. The sentence. The sentence."
    # --------------------------------------------------------------------------

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    cleaned_sentences = []

    for s in sentences:

        s = s.strip()

        if not s:
            continue

        if not cleaned_sentences:
            cleaned_sentences.append(s)
            continue

        if s.lower() == cleaned_sentences[-1].lower():
            continue

        cleaned_sentences.append(s)

    text = " ".join(cleaned_sentences)

    # --------------------------------------------------------------------------
    # Remove trailing generation artifacts
    # --------------------------------------------------------------------------

    text = re.split(
        r"\n\s*(?:<\|user\|>|<\|assistant\|>|Original:|Simplified:)",
        text,
        flags=re.IGNORECASE
    )[0]

    # --------------------------------------------------------------------------
    # Normalize whitespace
    # --------------------------------------------------------------------------

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ==============================================================================
# TEST PROMPT
# ==============================================================================

TEST_SENTENCE = originals[0]

TEST_PROMPT = cot_prompt(
    TEST_SENTENCE
)

print("=" * 100)
print("📝 TINYLLAMA CoT PROMPT TEST")
print("=" * 100)

print(TEST_PROMPT)

print("\n" + "=" * 100)
print("📊 PROMPT INFORMATION")
print("=" * 100)

print(
    f"Sample index : 0"
)

print(
    f"Input chars  : {len(TEST_PROMPT)}"
)

print(
    f"Input tokens : "
    f"{len(tokenizer(TEST_PROMPT)['input_ids'])}"
)

print("\n" + "=" * 100)
print("🧹 CLEAN FUNCTION TEST")
print("=" * 100)

test_junk = (
    "Original: Hello world.\n"
    "Simplified: Hello world. "
    "Hello world.<|endoftext|>"
)

print("Before:")
print(test_junk)

print("\nAfter:")
print(clean_prediction(test_junk))

print("\n" + "=" * 100)
print("✅ STEP 5 READY")
print("=" * 100)

📝 TINYLLAMA CoT PROMPT TEST
Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add new information.
Do not remove important information.

First, think carefully about the meaning of the sentence and identify
difficult words, phrases, or structures that should be simplified.

Then produce the final simplified sentence.

Do not show your reasoning.
Return only the final simplified sentence.

Original: Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

Simplified:

📊 PROMPT INFORMATION
Sample index : 0
Input chars  : 608
Input tokens : 151

🧹 CLEAN FUNCTION TEST
Before:
Original: Hello world.
Simplified: Hello world. Hello world.<|endoftext|>

After:
Hello world.

✅ STEP 5 READY


In [6]:
# ==============================================================================
# 🚀 STEP 6 — TINYLLAMA CoT SINGLE GENERATION TEST
# ==============================================================================

import time
import gc
import torch

SAMPLE_INDEX = 0

source = originals[SAMPLE_INDEX]

# ==============================================================================
# BUILD CoT PROMPT
# ==============================================================================

prompt = cot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# ==============================================================================
# APPLY CHAT TEMPLATE
# ==============================================================================

chat_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    chat_input,
    return_tensors="pt",
    truncation=True,
    max_length=2048
)

inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

input_tokens = inputs["input_ids"].shape[1]

# ==============================================================================
# VRAM RESET
# ==============================================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

# ==============================================================================
# GENERATION
# ==============================================================================

print("=" * 100)
print("🚀 TINYLLAMA — CoT SINGLE GENERATION TEST")
print("=" * 100)

print("\n🔵 ORIGINAL")
print("-" * 100)
print(source)

print("\n📝 CHAT TEMPLATE INPUT")
print("-" * 100)
print(chat_input)

print(f"\nInput tokens: {input_tokens}")

print("\n🚀 Generating...")

start_time = time.perf_counter()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,

        # ----------------------------------------------------------------------
        # ONLY max_new_tokens
        # ----------------------------------------------------------------------

        max_new_tokens=64,

        # ----------------------------------------------------------------------
        # Prevent padding issues
        # ----------------------------------------------------------------------

        pad_token_id=tokenizer.eos_token_id,

        # ----------------------------------------------------------------------
        # Deterministic evaluation
        # ----------------------------------------------------------------------

        do_sample=False,

        # ----------------------------------------------------------------------
        # Prevent endless repetition
        # ----------------------------------------------------------------------

        repetition_penalty=1.05,

        no_repeat_ngram_size=3,

        use_cache=True
    )

end_time = time.perf_counter()

# ==============================================================================
# DECODE ONLY GENERATED TOKENS
# ==============================================================================

generated_ids = output_ids[
    0,
    input_tokens:
]

raw_prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

# ==============================================================================
# CLEAN
# ==============================================================================

prediction = clean_prediction(
    raw_prediction
)

# ==============================================================================
# PERFORMANCE
# ==============================================================================

inference_time = end_time - start_time

if torch.cuda.is_available():

    peak_vram = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

else:

    peak_vram = 0.0

# ==============================================================================
# DISPLAY
# ==============================================================================

print("\n" + "=" * 100)
print("🤖 TINYLLAMA CoT OUTPUT")
print("=" * 100)

print(prediction)

print("\n" + "=" * 100)
print("⚡ PERFORMANCE")
print("=" * 100)

print(
    f"Inference time: {inference_time:.3f} sec"
)

print(
    f"Peak VRAM     : {peak_vram:.3f} GB"
)

print("\n" + "=" * 100)
print("✅ SINGLE CoT GENERATION TEST COMPLETE")
print("=" * 100)

# ==============================================================================
# CLEAN TEMPORARY TENSORS
# ==============================================================================

del inputs
del output_ids
del generated_ids

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

[transformers] Both `max_new_tokens` (=64) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 TINYLLAMA — CoT SINGLE GENERATION TEST

🔵 ORIGINAL
----------------------------------------------------------------------------------------------------
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

📝 CHAT TEMPLATE INPUT
----------------------------------------------------------------------------------------------------
<|user|>
Simplify the following sentence using clear and easy-to-understand language.

Preserve the original meaning.
Do not add new information.
Do not remove important information.

First, think carefully about the meaning of the sentence and identify
difficult words, phrases, or structures that should be simplified.

Then produce the final simplified sentence.

Do not show your reasoning.
Return only the final simplified sentence.

Original: Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solan

In [7]:
# ==============================================================================
# 🔧 STEP 6B — CLEAN GENERATION CONFIG
# ==============================================================================

print("=" * 100)
print("🔧 CLEANING TINYLLAMA GENERATION CONFIG")
print("=" * 100)

# Remove inherited max_length from generation configuration
if hasattr(model, "generation_config"):

    print("\nBefore:")
    print("max_length     :", model.generation_config.max_length)
    print("max_new_tokens :", model.generation_config.max_new_tokens)

    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = None

print("\nAfter:")
print("max_length     :", model.generation_config.max_length)
print("max_new_tokens :", model.generation_config.max_new_tokens)

print("\n" + "=" * 100)
print("✅ GENERATION CONFIG CLEANED")
print("=" * 100)

🔧 CLEANING TINYLLAMA GENERATION CONFIG

Before:
max_length     : 2048
max_new_tokens : None

After:
max_length     : None
max_new_tokens : None

✅ GENERATION CONFIG CLEANED


In [8]:
# ==============================================================================
# 🚀 STEP 6C — VERIFY CLEAN CoT GENERATION
# ==============================================================================

import time
import torch
import gc

SAMPLE_INDEX = 0

source = originals[SAMPLE_INDEX]

messages = [
    {
        "role": "user",
        "content": cot_prompt(source)
    }
]

chat_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    chat_input,
    return_tensors="pt",
    truncation=True,
    max_length=2048
)

inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

input_tokens = inputs["input_ids"].shape[1]

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("=" * 100)
print("🚀 TINYLLAMA — CLEAN CoT GENERATION TEST")
print("=" * 100)

print("\n🔵 ORIGINAL")
print("-" * 100)
print(source)

print("\nInput tokens:", input_tokens)
print("\n🚀 Generating...")

start_time = time.perf_counter()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
        use_cache=True
    )

end_time = time.perf_counter()

# ==============================================================================
# DECODE ONLY NEW TOKENS
# ==============================================================================

generated_ids = output_ids[0, input_tokens:]

raw_prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

prediction = clean_prediction(raw_prediction)

inference_time = end_time - start_time

if torch.cuda.is_available():
    peak_vram = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )
else:
    peak_vram = 0.0

# ==============================================================================
# DISPLAY
# ==============================================================================

print("\n" + "=" * 100)
print("🤖 TINYLLAMA CoT OUTPUT")
print("=" * 100)
print(prediction)

print("\n" + "=" * 100)
print("⚡ PERFORMANCE")
print("=" * 100)
print(f"Inference time: {inference_time:.3f} sec")
print(f"Peak VRAM     : {peak_vram:.3f} GB")

print("\n" + "=" * 100)
print("✅ CLEAN CoT GENERATION TEST COMPLETE")
print("=" * 100)

# ==============================================================================
# CLEAN MEMORY
# ==============================================================================

del inputs
del output_ids
del generated_ids

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

🚀 TINYLLAMA — CLEAN CoT GENERATION TEST

🔵 ORIGINAL
----------------------------------------------------------------------------------------------------
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

Input tokens: 166

🚀 Generating...

🤖 TINYLLAMA CoT OUTPUT
Adjacent county is Marin , Mendicino , Lake , Napa , Solano , and Contracosta .

⚡ PERFORMANCE
Inference time: 4.062 sec
Peak VRAM     : 2.126 GB

✅ CLEAN CoT GENERATION TEST COMPLETE


In [9]:
# ==============================================================================
# 🔧 STEP 7 — TINYLLAMA CoT METRICS
# ==============================================================================
# Correct handling:
#   1 prediction + 10 human references
#   Empty prediction protection
#   No ROUGE-L mismatch
# ==============================================================================

import evaluate
import textstat
import numpy as np
import gc

print("=" * 100)
print("🔧 LOADING EVALUATION METRICS")
print("=" * 100)

# ==============================================================================
# LOAD METRICS
# ==============================================================================

sari_metric = evaluate.load("sari")

bleu_metric = evaluate.load("bleu")

rouge_metric = evaluate.load("rouge")

bertscore_metric = evaluate.load("bertscore")

print("✅ Metrics loaded")


# ==============================================================================
# SAFE BLEU
# ==============================================================================

def safe_bleu(prediction, references):

    prediction = str(prediction).strip()

    references = [
        str(r).strip()
        for r in references
        if str(r).strip()
    ]

    # Empty prediction → BLEU = 0
    if not prediction:
        return 0.0

    # No references → BLEU = 0
    if not references:
        return 0.0

    try:

        result = bleu_metric.compute(
            predictions=[prediction],
            references=[references]
        )

        return float(result["bleu"])

    except ZeroDivisionError:

        return 0.0

    except Exception as e:

        print(f"⚠️ BLEU error: {e}")

        return 0.0


# ==============================================================================
# SAFE ROUGE-L
# ==============================================================================
# Evaluate's ROUGE expects prediction/reference pairs.
#
# We therefore calculate ROUGE-L against EACH of the 10 references
# separately and take the maximum score.
#
# This avoids:
#   Mismatch in the number of predictions (1) and references (10)
# ==============================================================================

def safe_rouge_l(prediction, references):

    prediction = str(prediction).strip()

    references = [
        str(r).strip()
        for r in references
        if str(r).strip()
    ]

    if not prediction or not references:
        return 0.0

    scores = []

    for reference in references:

        try:

            result = rouge_metric.compute(
                predictions=[prediction],
                references=[reference],
                rouge_types=["rougeL"]
            )

            scores.append(
                float(result["rougeL"])
            )

        except Exception as e:

            print(
                f"⚠️ ROUGE-L reference error: {e}"
            )

    if not scores:
        return 0.0

    return float(max(scores))


# ==============================================================================
# SAFE BERTSCORE
# ==============================================================================
# BERTScore is also calculated against all 10 references.
# We take the maximum F1, matching the best human reference.
# ==============================================================================

def safe_bertscore(prediction, references):

    prediction = str(prediction).strip()

    references = [
        str(r).strip()
        for r in references
        if str(r).strip()
    ]

    if not prediction or not references:
        return 0.0

    scores = []

    for reference in references:

        try:

            result = bertscore_metric.compute(
                predictions=[prediction],
                references=[reference],
                lang="en",
                model_type="roberta-large",
                rescale_with_baseline=False
            )

            scores.append(
                float(result["f1"][0])
            )

        except Exception as e:

            print(
                f"⚠️ BERTScore error: {e}"
            )

    if not scores:
        return 0.0

    return float(max(scores))


# ==============================================================================
# SAFE FKGL
# ==============================================================================

def safe_fkgl(prediction):

    prediction = str(prediction).strip()

    if not prediction:
        return 0.0

    try:

        return float(
            textstat.flesch_kincaid_grade(
                prediction
            )
        )

    except Exception as e:

        print(
            f"⚠️ FKGL error: {e}"
        )

        return 0.0


# ==============================================================================
# MAIN METRIC FUNCTION
# ==============================================================================

def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(prediction).strip()

    references = [
        str(r).strip()
        for r in references
    ]

    # --------------------------------------------------------------------------
    # SARI
    # --------------------------------------------------------------------------

    try:

        sari_result = sari_metric.compute(
            sources=[source],
            predictions=[prediction],
            references=[
                references
            ]
        )

        sari_score = float(
            sari_result["sari"]
        )

    except Exception as e:

        print(
            f"⚠️ SARI error: {e}"
        )

        sari_score = 0.0

    # --------------------------------------------------------------------------
    # BLEU
    # --------------------------------------------------------------------------

    bleu_score = safe_bleu(
        prediction,
        references
    )

    # --------------------------------------------------------------------------
    # ROUGE-L
    # --------------------------------------------------------------------------

    rouge_score = safe_rouge_l(
        prediction,
        references
    )

    # --------------------------------------------------------------------------
    # BERTScore-F1
    # --------------------------------------------------------------------------

    bert_score = safe_bertscore(
        prediction,
        references
    )

    # --------------------------------------------------------------------------
    # FKGL
    # --------------------------------------------------------------------------

    fkgl_score = safe_fkgl(
        prediction
    )

    return {

        "sari": sari_score,

        "bleu": bleu_score,

        "rougeL": rouge_score,

        "bertscore_f1": bert_score,

        "fkgl": fkgl_score
    }


print("\n" + "=" * 100)
print("✅ evaluate_one_sample() defined")
print("✅ Empty BLEU protection enabled")
print("✅ ROUGE-L 1-vs-10 mismatch fixed")
print("✅ BERTScore multi-reference handling enabled")
print("=" * 100)

🔧 LOADING EVALUATION METRICS
✅ Metrics loaded

✅ evaluate_one_sample() defined
✅ Empty BLEU protection enabled
✅ ROUGE-L 1-vs-10 mismatch fixed
✅ BERTScore multi-reference handling enabled


In [10]:
# ==============================================================================
# 📊 STEP 8 — SCORE TINYLLAMA CoT SAMPLE 0
# ==============================================================================

import time
import torch

SAMPLE_INDEX = 0

source = originals[SAMPLE_INDEX]
references = reference_groups[SAMPLE_INDEX]

# ------------------------------------------------------------------------------
# Use the clean prediction from STEP 6C
# ------------------------------------------------------------------------------

prediction = clean_prediction(
    prediction
)

print("=" * 100)
print("📊 SCORING TINYLLAMA CoT SAMPLE 0")
print("=" * 100)

print("\n🔵 ORIGINAL")
print("-" * 100)
print(source)

print("\n🤖 CLEANED PREDICTION")
print("-" * 100)
print(prediction)

print("\n📚 HUMAN REFERENCES")
print("-" * 100)

for i, ref in enumerate(references, start=1):
    print(f"{i}. {ref}")

# ------------------------------------------------------------------------------
# METRICS
# ------------------------------------------------------------------------------

test_scores = evaluate_one_sample(
    source,
    prediction,
    references
)

print("\n" + "=" * 100)
print("📊 METRICS")
print("-" * 100)

print(
    f"SARI          : {test_scores['sari']:.4f}"
)

print(
    f"BLEU          : {test_scores['bleu']:.4f}"
)

print(
    f"ROUGE-L       : {test_scores['rougeL']:.4f}"
)

print(
    f"BERTScore-F1  : {test_scores['bertscore_f1']:.4f}"
)

print(
    f"FKGL          : {test_scores['fkgl']:.4f}"
)

print("\n" + "=" * 100)
print("✅ SAMPLE 0 SCORING COMPLETE")
print("=" * 100)

📊 SCORING TINYLLAMA CoT SAMPLE 0

🔵 ORIGINAL
----------------------------------------------------------------------------------------------------
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

🤖 CLEANED PREDICTION
----------------------------------------------------------------------------------------------------
Adjacent county is Marin , Mendicino , Lake , Napa , Solano , and Contracosta .

📚 HUMAN REFERENCES
----------------------------------------------------------------------------------------------------
1. countries next to it are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.
2. Nearby counties are Marin, Mendocino, Lake, Napa, and Solano and Contra Costa.
3. Adjacent counties are Marin, Mendocino, Lake, Napa, Solano and Contra Costa.
4. Neighboring counties are Marin, Mendocino, Lake, Napa, Solano, and Contra Costa.
5. Adjacent counties are Marin (south), Mend

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 METRICS
----------------------------------------------------------------------------------------------------
SARI          : 31.4665
BLEU          : 0.5067
ROUGE-L       : 0.5714
BERTScore-F1  : 0.9274
FKGL          : 15.4500

✅ SAMPLE 0 SCORING COMPLETE


In [11]:
# ==============================================================================
# 🚀 TINYLLAMA — FULL CoT ASSET EVALUATION
# ==============================================================================
# 2,359 samples
# 10 ASSET references per sample
# Checkpoint every 25 samples
# Resume-safe
# ==============================================================================

import os
import gc
import time
import torch
import pandas as pd
import numpy as np

# ==============================================================================
# CONFIGURATION
# ==============================================================================

OUTPUT_FILE = "TINYLLAMA_COT_ASSET_2359.csv"

CHECKPOINT_EVERY = 25

MAX_NEW_TOKENS = 64

MAX_INPUT_TOKENS = 2048


# ==============================================================================
# HEADER
# ==============================================================================

print("=" * 100)
print("🚀 TINYLLAMA — FULL CoT ASSET EVALUATION")
print("=" * 100)

print(f"Total samples : {len(originals)}")
print(f"Output file   : {OUTPUT_FILE}")
print(f"Checkpoint    : every {CHECKPOINT_EVERY} samples")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print("=" * 100)


# ==============================================================================
# SAFETY CHECKS
# ==============================================================================

assert len(originals) == 2359, (
    f"Expected 2359 samples, found {len(originals)}"
)

assert len(reference_groups) == 2359, (
    f"Expected 2359 reference groups, found {len(reference_groups)}"
)

assert model is not None
assert tokenizer is not None
assert evaluate_one_sample is not None
assert clean_prediction is not None

print("✅ Model found")
print("✅ Tokenizer found")
print("✅ ASSET dataset found")
print("✅ Metric function found")
print("✅ Clean prediction function found")


# ==============================================================================
# LOAD EXISTING CSV IF AVAILABLE
# ==============================================================================

if os.path.exists(OUTPUT_FILE):

    print("\n🔄 Existing CSV found")

    results_df = pd.read_csv(
        OUTPUT_FILE,
        encoding="utf-8-sig"
    )

    print(
        f"Existing rows: {len(results_df)}"
    )

    # --------------------------------------------------------------------------
    # Identify completed sample indices
    # --------------------------------------------------------------------------

    if "sample_index" in results_df.columns:

        completed_ids = set(
            pd.to_numeric(
                results_df["sample_index"],
                errors="coerce"
            )
            .dropna()
            .astype(int)
            .tolist()
        )

    else:

        # Fallback if old CSV does not contain sample_index
        completed_ids = set(
            range(len(results_df))
        )

else:

    print("\n🆕 Starting new evaluation")

    results_df = pd.DataFrame()

    completed_ids = set()


# ==============================================================================
# REMOVE INVALID IDS
# ==============================================================================

completed_ids = {
    i for i in completed_ids
    if 0 <= i < len(originals)
}


print(
    f"Completed samples: {len(completed_ids)}"
)

print(
    f"Remaining samples: {len(originals) - len(completed_ids)}"
)


# ==============================================================================
# FULL EVALUATION LOOP
# ==============================================================================

total_samples = len(originals)


for sample_index in range(total_samples):

    # --------------------------------------------------------------------------
    # RESUME SUPPORT
    # --------------------------------------------------------------------------

    if sample_index in completed_ids:
        continue


    sample_number = sample_index + 1

    source = originals[sample_index]

    references = reference_groups[sample_index]


    # ==========================================================================
    # DISPLAY SAMPLE
    # ==========================================================================

    print("\n" + "=" * 100)
    print(
        f"🧪 TINYLLAMA CoT — SAMPLE "
        f"{sample_number}/{total_samples}"
    )
    print("=" * 100)

    print("\n🔵 ORIGINAL")
    print("-" * 100)
    print(source)


    # ==========================================================================
    # BUILD CoT CHAT PROMPT
    # ==========================================================================

    prompt = cot_prompt(source)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # ==========================================================================
    # TOKENIZE
    # ==========================================================================

    inputs = tokenizer(
        chat_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]


    # ==========================================================================
    # RESET CUDA MEMORY STATS
    # ==========================================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # ==========================================================================
    # GENERATION
    # ==========================================================================

    start_time = time.perf_counter()

    try:

        with torch.inference_mode():

            output_ids = model.generate(

                **inputs,

                # --------------------------------------------------------------
                # ONLY max_new_tokens
                # --------------------------------------------------------------

                max_new_tokens=MAX_NEW_TOKENS,

                # --------------------------------------------------------------
                # Deterministic generation
                # --------------------------------------------------------------

                do_sample=False,

                # --------------------------------------------------------------
                # Padding
                # --------------------------------------------------------------

                pad_token_id=tokenizer.eos_token_id,

                # --------------------------------------------------------------
                # Repetition control
                # --------------------------------------------------------------

                repetition_penalty=1.05,

                no_repeat_ngram_size=3,

                use_cache=True
            )


        end_time = time.perf_counter()

        inference_time = (
            end_time - start_time
        )


        # ======================================================================
        # DECODE ONLY GENERATED TOKENS
        # ======================================================================

        generated_ids = output_ids[
            0,
            input_tokens:
        ]

        raw_prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )


        # ======================================================================
        # CLEAN PREDICTION
        # ======================================================================

        prediction = clean_prediction(
            raw_prediction
        )


        # ======================================================================
        # VRAM
        # ======================================================================

        if torch.cuda.is_available():

            peak_vram = (
                torch.cuda.max_memory_allocated()
                / 1024**3
            )

        else:

            peak_vram = 0.0


        # ======================================================================
        # DISPLAY PREDICTION
        # ======================================================================

        print("\n🤖 PREDICTION")
        print("-" * 100)
        print(prediction)


        # ======================================================================
        # METRICS
        # ======================================================================

        try:

            scores = evaluate_one_sample(
                source,
                prediction,
                references
            )

        except Exception as metric_error:

            print("\n⚠️ METRIC ERROR")
            print(metric_error)

            # --------------------------------------------------------------
            # Never destroy the whole evaluation because of one sample
            # --------------------------------------------------------------

            scores = {
                "sari": 0.0,
                "bleu": 0.0,
                "rougeL": 0.0,
                "bertscore_f1": 0.0,
                "fkgl": 0.0
            }


        # ======================================================================
        # DISPLAY METRICS
        # ======================================================================

        print("\n📊 METRICS")
        print("-" * 100)

        print(
            f"SARI          : "
            f"{scores['sari']:.4f}"
        )

        print(
            f"BLEU          : "
            f"{scores['bleu']:.4f}"
        )

        print(
            f"ROUGE-L       : "
            f"{scores['rougeL']:.4f}"
        )

        print(
            f"BERTScore-F1  : "
            f"{scores['bertscore_f1']:.4f}"
        )

        print(
            f"FKGL          : "
            f"{scores['fkgl']:.4f}"
        )


        # ======================================================================
        # PERFORMANCE
        # ======================================================================

        print("\n⚡ PERFORMANCE")
        print("-" * 100)

        print(
            f"Inference time: "
            f"{inference_time:.3f} sec"
        )

        print(
            f"Peak VRAM     : "
            f"{peak_vram:.3f} GB"
        )


        # ======================================================================
        # CREATE CSV ROW
        # ======================================================================

        row = {

            "sample_index":
                sample_index,

            "source":
                source,

            "prediction":
                prediction,

            "references":
                " ||| ".join(
                    references
                ),

            "sari":
                scores["sari"],

            "bleu":
                scores["bleu"],

            "rougeL":
                scores["rougeL"],

            "bertscore_f1":
                scores["bertscore_f1"],

            "fkgl":
                scores["fkgl"],

            "inference_time_sec":
                inference_time,

            "peak_vram_gb":
                peak_vram
        }


        # ======================================================================
        # APPEND ROW
        # ======================================================================

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([row])
            ],
            ignore_index=True
        )


        completed_ids.add(
            sample_index
        )


        # ======================================================================
        # CHECKPOINT SAVE
        # ======================================================================

        completed_now = len(completed_ids)

        if (
            completed_now % CHECKPOINT_EVERY == 0
        ):

            results_df.to_csv(
                OUTPUT_FILE,
                index=False,
                encoding="utf-8-sig"
            )

            print("\n" + "=" * 100)
            print("💾 CHECKPOINT SAVED")
            print("=" * 100)

            print(
                f"Completed : "
                f"{completed_now}/{total_samples}"
            )

            print(
                f"Remaining : "
                f"{total_samples - completed_now}"
            )

            print(
                f"File      : "
                f"{OUTPUT_FILE}"
            )

            print("=" * 100)


        # ======================================================================
        # MEMORY CLEANUP
        # ======================================================================

        del inputs
        del output_ids
        del generated_ids

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        # ======================================================================
        # SAMPLE COMPLETE
        # ======================================================================

        print("\n" + "=" * 100)

        print(
            f"✅ SAMPLE {sample_number} COMPLETE"
        )

        print("=" * 100)


    # ==========================================================================
    # GENERATION ERROR
    # ==========================================================================

    except Exception as generation_error:

        print("\n" + "=" * 100)
        print("❌ GENERATION ERROR")
        print("=" * 100)

        print(
            generation_error
        )

        print(
            f"Sample index: {sample_index}"
        )

        print(
            "⚠️ Sample was NOT added to the completed set."
        )

        print(
            "The loop will stop so the problem can be inspected."
        )

        raise


# ==============================================================================
# FINAL SAVE
# ==============================================================================

results_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ==============================================================================
# FINAL REPORT
# ==============================================================================

print("\n" + "=" * 100)
print("🎉 TINYLLAMA CoT EVALUATION COMPLETE")
print("=" * 100)

print(
    f"Total samples : "
    f"{len(results_df)}"
)

print(
    f"Output file   : "
    f"{OUTPUT_FILE}"
)

print("=" * 100)

🚀 TINYLLAMA — FULL CoT ASSET EVALUATION
Total samples : 2359
Output file   : TINYLLAMA_COT_ASSET_2359.csv
Checkpoint    : every 25 samples
Max new tokens: 64
✅ Model found
✅ Tokenizer found
✅ ASSET dataset found
✅ Metric function found
✅ Clean prediction function found

🔄 Existing CSV found
Existing rows: 25
Completed samples: 25
Remaining samples: 2334

🧪 TINYLLAMA CoT — SAMPLE 26/2359

🔵 ORIGINAL
----------------------------------------------------------------------------------------------------
The British Rail Class 91 is a class of 140 mph (225 km / h), 4.54MW / 6,300 hp electric locomotives ordered as a component of the East Coast Main Line modernisation and electrification programme of the late 1980s.

🤖 PREDICTION
----------------------------------------------------------------------------------------------------
The British Rail class 90 is a type of electric locomotive . It was built by British Railways from 1897 to 1200 .

📊 METRICS
------------------------------------------

In [12]:
# ============================================================
# TINYLLAMA CoT — DESCRIPTIVE STATISTICS
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# FILE
# ------------------------------------------------------------

CSV_FILE = "TINYLLAMA_COT_ASSET_2359.csv"

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"❌ File not found: {CSV_FILE}")

df = pd.read_csv(CSV_FILE)

print("=" * 100)
print("📊 TINYLLAMA CoT — DESCRIPTIVE STATISTICS")
print("=" * 100)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# ------------------------------------------------------------
# SHOW COLUMNS
# ------------------------------------------------------------

print("\n📋 COLUMNS")
print("-" * 100)

for col in df.columns:
    print(col)

# ------------------------------------------------------------
# METRICS TO ANALYZE
# ------------------------------------------------------------

metric_candidates = [
    "sari",
    "bleu",
    "rouge",
    "rouge_l",
    "rouge-l",
    "bertscore",
    "bertscore_f1",
    "bertscore-f1",
    "fkgl",
    "inference_time",
    "inference time",
    "peak_vram",
    "peak vram"
]

# Match columns case-insensitively
selected_columns = []

for col in df.columns:
    normalized = col.lower().strip()

    if normalized in metric_candidates:
        selected_columns.append(col)

# ------------------------------------------------------------
# IF COLUMN NAMES DIFFER, DETECT NUMERIC COLUMNS
# ------------------------------------------------------------

if not selected_columns:

    print("\n⚠️ Exact metric names were not detected.")
    print("Looking for numeric columns instead...")

    selected_columns = df.select_dtypes(
        include=[np.number]
    ).columns.tolist()

print("\n✅ ANALYZING:")
for col in selected_columns:
    print("   •", col)

# ------------------------------------------------------------
# CONVERT TO NUMERIC
# ------------------------------------------------------------

for col in selected_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

stats = pd.DataFrame({
    "Metric": selected_columns,
    "Mean": [df[col].mean() for col in selected_columns],
    "Median": [df[col].median() for col in selected_columns],
    "Min": [df[col].min() for col in selected_columns],
    "Max": [df[col].max() for col in selected_columns],
    "Std": [df[col].std() for col in selected_columns],
    "Valid": [df[col].notna().sum() for col in selected_columns],
    "Missing": [df[col].isna().sum() for col in selected_columns]
})

# ------------------------------------------------------------
# ROUND VALUES
# ------------------------------------------------------------

stats[["Mean", "Median", "Min", "Max", "Std"]] = \
    stats[["Mean", "Median", "Min", "Max", "Std"]].round(6)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n")
print("=" * 100)
print("📊 MEAN / MEDIAN / MIN / MAX / STD")
print("=" * 100)

print(stats.to_string(index=False))

# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_FILE = "TINYLLAMA_COT_DESCRIPTIVE_STATISTICS.csv"

stats.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 100)
print("✅ STATISTICS COMPLETE")
print("=" * 100)

print(f"📁 Saved to: {OUTPUT_FILE}")

📊 TINYLLAMA CoT — DESCRIPTIVE STATISTICS
Total rows: 2359
Total columns: 11

📋 COLUMNS
----------------------------------------------------------------------------------------------------
sample_index
source
prediction
references
sari
bleu
rougeL
bertscore_f1
fkgl
inference_time_sec
peak_vram_gb

✅ ANALYZING:
   • sari
   • bleu
   • bertscore_f1
   • fkgl


📊 MEAN / MEDIAN / MIN / MAX / STD
      Metric      Mean    Median       Min       Max      Std  Valid  Missing
        sari 29.338778 28.466342 11.422030 72.175679 6.810112   2359        0
        bleu  0.188131  0.168325  0.000000  1.000000 0.209501   2359        0
bertscore_f1  0.922194  0.924010  0.817427  0.984078 0.024161   2359        0
        fkgl  8.320515  8.350000 -1.840000 24.100000 3.578619   2359        0

✅ STATISTICS COMPLETE
📁 Saved to: TINYLLAMA_COT_DESCRIPTIVE_STATISTICS.csv
